# Database Explorer
Loads top 100 rows from every table in every SQLite database found under the project root.
Use this as a starting point for ETL and reporting — swap `LIMIT` or filter `DB_PATHS` as needed.

In [47]:
import sqlite3
import pandas as pd
from pathlib import Path

# ── Config ────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path("../")          # relative to this notebook
ROW_LIMIT    = 100

# Auto-discover all .db / .sqlite files, or pin specific paths:
# DB_PATHS = [PROJECT_ROOT / "database/pipeline.db"]
DB_PATHS = sorted(
    list(PROJECT_ROOT.rglob("*.db")) +
    list(PROJECT_ROOT.rglob("*.sqlite"))
)

print(f"Found {len(DB_PATHS)} database(s):")
for p in DB_PATHS:
    print(f"  {p}")

Found 1 database(s):
  ..\database\pipeline.db


In [48]:
def load_db(path: Path, limit: int) -> dict[str, pd.DataFrame]:
    """Return {table_name: DataFrame} for every user table in the database."""
    conn = sqlite3.connect(path)
    tables = pd.read_sql(
        "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'",
        conn
    )["name"].tolist()
    result = {}
    for table in tables:
        result[table] = pd.read_sql(f'SELECT * FROM "{table}" LIMIT {limit}', conn)
    conn.close()
    return result

# ── Load all databases ─────────────────────────────────────────────────────────
dbs: dict[str, dict[str, pd.DataFrame]] = {}

for db_path in DB_PATHS:
    key = str(db_path.relative_to(PROJECT_ROOT))
    dbs[key] = load_db(db_path, ROW_LIMIT)
    for table, df in dbs[key].items():
        print(f"{key}  →  {table}  ({len(df)} rows, {len(df.columns)} cols)")

database\pipeline.db  →  processed_documents  (100 rows, 17 cols)
database\pipeline.db  →  review_queue  (46 rows, 7 cols)
database\pipeline.db  →  template_stats  (100 rows, 6 cols)
database\pipeline.db  →  parsed_invoices  (28 rows, 24 cols)
database\pipeline.db  →  invoice_lines  (74 rows, 26 cols)
database\pipeline.db  →  contacts  (1 rows, 8 cols)


## pipeline.db — processed_documents

In [49]:
df = dbs["database\pipeline.db"]["processed_documents"]
df

<>:1: SyntaxWarning: invalid escape sequence '\p'
<>:1: SyntaxWarning: invalid escape sequence '\p'
C:\Users\joelw\AppData\Local\Temp\ipykernel_22724\3925258186.py:1: SyntaxWarning: invalid escape sequence '\p'
  df = dbs["database\pipeline.db"]["processed_documents"]


,id,message_id,attachment_filename,sender_email,received_at,customer_name,invoice_number,confidence,needs_review,json_path,raw_text_path,attachment_path,processed_at,error,status,template_name,content_hash
0,1,AQMkADAwATNiZmYAZC1kNTczLTMyOTgtMDACLTAwCgBGAA...,Invoice INV-8388.pdf,rhettalongbeach@email.propertyme.com,2026-04-21T22:42:10Z,NaN,NaN,0.00,0,NaN,None,NaN,2026-04-22T12:32:20.301499Z,BLOCKED: Sender domain not in allowlist: rhett...,None,None,None
1,2,AQMkADAwATNiZmYAZC1kNTczLTMyOTgtMDACLTAwCgBGAA...,Mobile 22 Apr 060759.jpg,rhettalongbeach@email.propertyme.com,2026-04-21T22:42:10Z,NaN,NaN,0.00,0,NaN,None,NaN,2026-04-22T12:32:20.313110Z,BLOCKED: Sender domain not in allowlist: rhett...,None,None,None
2,3,AQMkADAwATNiZmYAZC1kNTczLTMyOTgtMDACLTAwCgBGAA...,Mobile 22 Apr 060806.jpg,rhettalongbeach@email.propertyme.com,2026-04-21T22:42:10Z,NaN,NaN,0.00,0,NaN,None,NaN,2026-04-22T12:32:20.323951Z,BLOCKED: Sender domain not in allowlist: rhett...,None,None,None
3,4,AQMkADAwATNiZmYAZC1kNTczLTMyOTgtMDACLTAwCgBGAA...,007163C1.pdf,donotreply@stratamax.com.au,2026-04-21T22:50:38Z,NaN,NaN,0.00,0,NaN,None,NaN,2026-04-22T12:32:24.037402Z,BLOCKED: Sender domain not in allowlist: donot...,None,None,None
4,5,AQMkADAwATNiZmYAZC1kNTczLTMyOTgtMDACLTAwCgBGAA...,image0.jpeg,bonitahua@hotmail.com,2026-04-21T23:18:24Z,NaN,NaN,0.00,0,NaN,None,NaN,2026-04-22T12:32:28.246942Z,BLOCKED: Sender domain not in allowlist: bonit...,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,AQMkADAwATNiZmYAZC1kNTczLTMyOTgtMDACLTAwCgBGAA...,MAR26 - INV281931 - 415425115374.pdf,joelwu95@outlook.com,2026-04-23T12:05:25Z,Joel Wu,NaN,0.98,1,C:\git\ms_outlook\parsed\AQMkADAwATNiZmYA\MAR2...,None,C:\git\ms_outlook\attachments\AQMkADAwATNiZmYA...,2026-04-23T12:34:19.669549Z,NaN,None,None,None
96,97,AQMkADAwATNiZmYAZC1kNTczLTMyOTgtMDACLTAwCgBGAA...,image0.jpeg,bonitahua@hotmail.com,2026-04-21T23:18:24Z,Bonita Hua,NaN,0.98,1,C:\git\ms_outlook\parsed\AQMkADAwATNiZmYA\imag...,None,C:\git\ms_outlook\attachments\AQMkADAwATNiZmYA...,2026-04-23T12:37:19.480910Z,NaN,None,None,None
97,98,AQMkADAwATNiZmYAZC1kNTczLTMyOTgtMDACLTAwCgBGAA...,image1.jpeg,bonitahua@hotmail.com,2026-04-21T23:18:24Z,Bonita Hua,NaN,0.98,1,C:\git\ms_outlook\parsed\AQMkADAwATNiZmYA\imag...,None,C:\git\ms_outlook\attachments\AQMkADAwATNiZmYA...,2026-04-23T12:37:21.969910Z,NaN,None,None,None
98,99,AQMkADAwATNiZmYAZC1kNTczLTMyOTgtMDACLTAwCgBGAA...,image2.jpeg,bonitahua@hotmail.com,2026-04-21T23:18:24Z,Bonita Hua,NaN,0.98,1,C:\git\ms_outlook\parsed\AQMkADAwATNiZmYA\imag...,None,C:\git\ms_outlook\attachments\AQMkADAwATNiZmYA...,2026-04-23T12:37:27.780820Z,NaN,None,None,None


## pipeline.db — review_queue

In [50]:
df = dbs["database\pipeline.db"]["review_queue"]
df

<>:1: SyntaxWarning: invalid escape sequence '\p'
<>:1: SyntaxWarning: invalid escape sequence '\p'
C:\Users\joelw\AppData\Local\Temp\ipykernel_22724\1629862423.py:1: SyntaxWarning: invalid escape sequence '\p'
  df = dbs["database\pipeline.db"]["review_queue"]


,id,processed_doc_id,status,reason,created_at,resolved_at,resolved_by
0,1,179,pending,extracted_only,2026-04-25T10:19:47.798865Z,None,None
1,2,180,pending,extracted_only,2026-04-25T10:19:48.764180Z,None,None
2,3,181,pending,extracted_only,2026-04-25T10:19:48.863620Z,None,None
3,4,182,pending,extracted_only,2026-04-25T10:19:48.981415Z,None,None
4,5,183,pending,extracted_only,2026-04-25T10:19:49.267952Z,None,None
5,6,184,pending,extracted_only,2026-04-25T10:19:50.860894Z,None,None
6,7,185,pending,extracted_only,2026-04-25T10:19:51.689499Z,None,None
7,8,186,pending,extracted_only,2026-04-25T10:19:51.885804Z,None,None
8,9,187,pending,extracted_only,2026-04-25T10:19:52.039698Z,None,None
9,10,188,pending,extracted_only,2026-04-25T10:19:52.210708Z,None,None


## pipeline.db — template_stats

In [51]:
df = dbs["database\pipeline.db"]["template_stats"]
df

<>:1: SyntaxWarning: invalid escape sequence '\p'
<>:1: SyntaxWarning: invalid escape sequence '\p'
C:\Users\joelw\AppData\Local\Temp\ipykernel_22724\3273322209.py:1: SyntaxWarning: invalid escape sequence '\p'
  df = dbs["database\pipeline.db"]["template_stats"]


,id,template_name,run_at,confidence,required_fields_matched,required_fields_total
0,1,topcut,2026-04-25T12:06:19.546596Z,1.00,4,3
1,2,topcut,2026-04-25T12:06:19.555350Z,1.00,9,8
2,3,topcut,2026-04-25T12:06:19.598047Z,0.00,0,3
3,4,topcut,2026-04-25T12:06:19.605721Z,0.00,1,8
4,5,topcut,2026-04-25T12:06:20.435545Z,1.00,4,3
...,...,...,...,...,...,...
95,96,topcut,2026-04-26T12:56:47.928825Z,0.00,0,3
96,97,topcut,2026-04-26T12:56:47.959623Z,0.00,1,8
97,98,topcut,2026-04-26T13:09:31.274531Z,1.00,4,3
98,99,topcut,2026-04-26T13:09:31.282160Z,1.00,9,8


## pipeline.db — parsed_invoices

In [52]:
df = dbs["database\pipeline.db"]["parsed_invoices"]
df

<>:1: SyntaxWarning: invalid escape sequence '\p'
<>:1: SyntaxWarning: invalid escape sequence '\p'
C:\Users\joelw\AppData\Local\Temp\ipykernel_22724\1273207791.py:1: SyntaxWarning: invalid escape sequence '\p'
  df = dbs["database\pipeline.db"]["parsed_invoices"]


,id,source_file,synced_at,customer_name,abn,address,order_date,requested_delivery_date,invoice_number,po_number,...,tax_amount,total_amount,line_items,message_id,sender_email,received_at,confidence,status,needs_review,processed_at
0,1,C:\git\ms_outlook\attachments\2026-04-24_topcu...,2026-04-28T07:12:36.283769Z,Paulo Penecitos,None,None,23.04.2026,None,None,22180125712401UDWM09,...,NaN,705.03,"[{""product_code"": ""100793"", ""description"": ""Bu...",AQMkADAwATNiZmYAZC1kNTczLTMyOTgtMDACLTAwCgBGAA...,Paulo.Penecitos@topcut.com.au,2026-04-24T04:30:27Z,0.97,parsed,0,2026-04-26T13:16:42.001054Z
1,2,C:\git\ms_outlook\attachments\2026-04-24_topcu...,2026-04-28T07:12:36.319806Z,Paulo Penecitos,None,None,"Thursday, 23 April 2026",None,None,20260423212012,...,NaN,NaN,"[{""product_code"": ""16027"", ""description"": ""Top...",AQMkADAwATNiZmYAZC1kNTczLTMyOTgtMDACLTAwCgBGAA...,Paulo.Penecitos@topcut.com.au,2026-04-24T04:30:27Z,0.85,parsed,0,2026-04-26T13:16:41.663520Z
2,3,C:\git\ms_outlook\attachments\2026-04-24_topcu...,2026-04-28T07:12:36.355796Z,Paulo Penecitos,None,None,NaN,None,None,3685450,...,NaN,NaN,"[{""product_code"": ""48377"", ""description"": ""*P*...",AQMkADAwATNiZmYAZC1kNTczLTMyOTgtMDACLTAwCgBGAA...,Paulo.Penecitos@topcut.com.au,2026-04-24T04:30:27Z,0.85,parsed,0,2026-04-26T13:16:40.503366Z
3,4,C:\git\ms_outlook\attachments\2026-04-24_topcu...,2026-04-28T07:12:36.391918Z,Paulo Penecitos,None,None,23/Apr/26,None,None,230426-BORMJA,...,0.00,"1,187.54","[{""product_code"": ""600237"", ""description"": ""BA...",AQMkADAwATNiZmYAZC1kNTczLTMyOTgtMDACLTAwCgBGAA...,Paulo.Penecitos@topcut.com.au,2026-04-24T04:30:27Z,0.97,parsed,0,2026-04-26T13:16:40.464192Z
4,5,C:\git\ms_outlook\attachments\2026-04-24_topcu...,2026-04-28T07:12:36.423519Z,Paulo Penecitos,None,None,23/04/2026,None,None,F52832261,...,NaN,NaN,"[{""product_code"": ""48245"", ""description"": ""Por...",AQMkADAwATNiZmYAZC1kNTczLTMyOTgtMDACLTAwCgBGAA...,Paulo.Penecitos@topcut.com.au,2026-04-24T04:30:27Z,0.85,parsed,0,2026-04-26T13:16:41.775780Z
5,6,C:\git\ms_outlook\attachments\2026-04-24_topcu...,2026-04-28T07:12:36.457272Z,Paulo Penecitos,None,None,23-APR-26,None,None,7760180,...,0.00,"1,149.50","[{""product_code"": ""15196"", ""description"": ""Bee...",AQMkADAwATNiZmYAZC1kNTczLTMyOTgtMDACLTAwCgBGAA...,Paulo.Penecitos@topcut.com.au,2026-04-24T04:30:27Z,0.97,parsed,0,2026-04-26T13:16:41.433926Z
6,7,C:\git\ms_outlook\attachments\2026-04-24_topcu...,2026-04-28T07:12:36.490823Z,Paulo Penecitos,None,None,23.04.2026,None,None,22180125712401UDWM09,...,NaN,705.03,"[{""product_code"": ""100793"", ""description"": ""Bu...",AQMkADAwATNiZmYAZC1kNTczLTMyOTgtMDACLTAwCgBGAA...,Paulo.Penecitos@topcut.com.au,2026-04-24T04:30:27Z,0.97,parsed,0,2026-04-26T12:56:41.106828Z
7,8,C:\git\ms_outlook\attachments\2026-04-24_topcu...,2026-04-28T07:12:36.523675Z,Paulo Penecitos,None,None,"Thursday, 23 April 2026",None,None,20260423212012,...,NaN,NaN,"[{""product_code"": ""16027"", ""description"": ""Bee...",AQMkADAwATNiZmYAZC1kNTczLTMyOTgtMDACLTAwCgBGAA...,Paulo.Penecitos@topcut.com.au,2026-04-24T04:30:27Z,0.85,parsed,0,2026-04-26T12:56:40.716051Z
8,9,C:\git\ms_outlook\attachments\2026-04-24_topcu...,2026-04-28T07:12:36.556347Z,Paulo Penecitos,None,None,NaN,None,None,3685450,...,NaN,NaN,"[{""product_code"": ""48377"", ""description"": ""*P*...",AQMkADAwATNiZmYAZC1kNTczLTMyOTgtMDACLTAwCgBGAA...,Paulo.Penecitos@topcut.com.au,2026-04-24T04:30:27Z,0.85,parsed,0,2026-04-26T12:56:39.523475Z
9,10,C:\git\ms_outlook\attachments\2026-04-24_topcu...,2026-04-28T07:12:36.586477Z,Paulo Penecitos,None,None,23/Apr/26,None,None,230426-BORMJA,...,0.00,"1,187.54","[{""product_code"": ""600237"", ""description"": ""BA...",AQMkADAwATNiZmYAZC1kNTczLTMyOTgtMDACLTAwCgBGAA...,Paulo.Penecitos@topcut.com.au,2026-04-24T04:30:27Z,0.97,parsed,0,2026-04-26T12:56:39.478001Z


## pipeline.db — invoice_lines

In [53]:
df = dbs["database\pipeline.db"]["invoice_lines"]
df

<>:1: SyntaxWarning: invalid escape sequence '\p'
<>:1: SyntaxWarning: invalid escape sequence '\p'
C:\Users\joelw\AppData\Local\Temp\ipykernel_22724\2874800246.py:1: SyntaxWarning: invalid escape sequence '\p'
  df = dbs["database\pipeline.db"]["invoice_lines"]


,id,source_file,line_num,message_id,sender_email,customer_name,company_name,company_abn,po_number,invoice_number,...,status,received_at,processed_at,product_code,description,qty,uom,unit_price,line_subtotal,line_total
0,1,C:\git\ms_outlook\attachments\2026-04-24_topcu...,1,AQMkADAwATNiZmYAZC1kNTczLTMyOTgtMDACLTAwCgBGAA...,Paulo.Penecitos@topcut.com.au,Paulo Penecitos,voco Gold Coast,35 609 038 502,22180125712401UDWM09,None,...,parsed,2026-04-24T04:30:27Z,2026-04-26T13:16:42.001054Z,100793,"Burger patty, wagyu beef, Laneway, low p",4.000,piece/unit,71.26,NaN,285.03
1,2,C:\git\ms_outlook\attachments\2026-04-24_topcu...,2,AQMkADAwATNiZmYAZC1kNTczLTMyOTgtMDACLTAwCgBGAA...,Paulo.Penecitos@topcut.com.au,Paulo Penecitos,voco Gold Coast,35 609 038 502,22180125712401UDWM09,None,...,parsed,2026-04-24T04:30:27Z,2026-04-26T13:16:42.001054Z,500873,"Lamb, diced, 25x25mm, multiple wrap, va",15.000,kilogram,28.00,NaN,420.00
2,3,C:\git\ms_outlook\attachments\2026-04-24_topcu...,1,AQMkADAwATNiZmYAZC1kNTczLTMyOTgtMDACLTAwCgBGAA...,Paulo.Penecitos@topcut.com.au,Paulo Penecitos,TGI Friday's Robina,NaN,20260423212012,None,...,parsed,2026-04-24T04:30:27Z,2026-04-26T13:16:41.663520Z,16027,Top Cut Beef Cube Roll 6-Star 280Gm,10,Each,NaN,NaN,NaN
3,4,C:\git\ms_outlook\attachments\2026-04-24_topcu...,2,AQMkADAwATNiZmYAZC1kNTczLTMyOTgtMDACLTAwCgBGAA...,Paulo.Penecitos@topcut.com.au,Paulo Penecitos,TGI Friday's Robina,NaN,20260423212012,None,...,parsed,2026-04-24T04:30:27Z,2026-04-26T13:16:41.663520Z,17973,Top Cut Beef Rib Eye Unfrenched 350Gm,6,EA,NaN,NaN,NaN
4,5,C:\git\ms_outlook\attachments\2026-04-24_topcu...,3,AQMkADAwATNiZmYAZC1kNTczLTMyOTgtMDACLTAwCgBGAA...,Paulo.Penecitos@topcut.com.au,Paulo Penecitos,TGI Friday's Robina,NaN,20260423212012,None,...,parsed,2026-04-24T04:30:27Z,2026-04-26T13:16:41.663520Z,13432,Top Cut Beef Striploin 250G Grain Fed,35,Each,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69,70,C:\git\ms_outlook\attachments\2026-04-24_topcu...,1,AQMkADAwATNiZmYAZC1kNTczLTMyOTgtMDACLTAwCgBGAA...,Paulo.Penecitos@topcut.com.au,Paulo Penecitos,Bavarian Bier Cafe,NaN,F52832261,None,...,parsed,2026-04-24T04:29:57Z,2026-04-28T07:32:11.683319Z,15335,5 RUMP PC 250g MV Chilled 4kg only please,4,Kg,NaN,NaN,NaN
70,71,C:\git\ms_outlook\attachments\2026-04-24_topcu...,2,AQMkADAwATNiZmYAZC1kNTczLTMyOTgtMDACLTAwCgBGAA...,Paulo.Penecitos@topcut.com.au,Paulo Penecitos,Bavarian Bier Cafe,NaN,F52832261,None,...,parsed,2026-04-24T04:29:57Z,2026-04-28T07:32:11.683319Z,15329,Beef 5* Rump PC 200G MV CHL 4kg only please. T...,4,Kg,NaN,NaN,NaN
71,72,C:\git\ms_outlook\attachments\2026-04-24_topcu...,3,AQMkADAwATNiZmYAZC1kNTczLTMyOTgtMDACLTAwCgBGAA...,Paulo.Penecitos@topcut.com.au,Paulo Penecitos,Bavarian Bier Cafe,NaN,F52832261,None,...,parsed,2026-04-24T04:29:57Z,2026-04-28T07:32:11.683319Z,48245,Pork - Schnitzel Grain Fed,4,Kg,NaN,NaN,NaN
72,73,C:\git\ms_outlook\attachments\2026-04-26_topcu...,1,AQMkADAwATNiZmYAZC1kNTczLTMyOTgtMDACLTAwCgBGAA...,Paulo.Penecitos@topcut.com.au,Paulo Penecitos,NaN,NaN,NaN,None,...,low_confidence,2026-04-26T11:32:41Z,2026-04-28T07:32:26.790610Z,48377,7 kg *P* MINCE COARSE 10MM VAC X 2.5KG CHL,3685450,NaN,NaN,NaN,NaN


## pipeline.db — contacts

In [54]:
df = dbs["database\pipeline.db"]["contacts"]
df

<>:1: SyntaxWarning: invalid escape sequence '\p'
<>:1: SyntaxWarning: invalid escape sequence '\p'
C:\Users\joelw\AppData\Local\Temp\ipykernel_22724\1586802595.py:1: SyntaxWarning: invalid escape sequence '\p'
  df = dbs["database\pipeline.db"]["contacts"]


,id,name,domains,emails,abns,keywords,template,last_synced_at
0,1,Paulo Penecitos,"[""topcut.com.au""]",[],[],"[""top cut"", ""topcut""]",topcut,2026-04-28T07:32:04.204275Z


---
## Template — custom query
Copy this cell to build your own queries for ETL / reporting.

In [60]:
import sqlite3, pandas as pd
from pathlib import Path

conn = sqlite3.connect(Path("../database/pipeline.db"))

query = """
SELECT *
FROM invoice_lines
WHERE po_number = '22180125712401UDWM09'
ORDER BY processed_at DESC
LIMIT 100
"""

df = pd.read_sql(query, conn)
conn.close()
df

,id,source_file,line_num,message_id,sender_email,customer_name,company_name,company_abn,po_number,invoice_number,...,status,received_at,processed_at,product_code,description,qty,uom,unit_price,line_subtotal,line_total
0,1,C:\git\ms_outlook\attachments\2026-04-24_topcu...,1,AQMkADAwATNiZmYAZC1kNTczLTMyOTgtMDACLTAwCgBGAA...,Paulo.Penecitos@topcut.com.au,Paulo Penecitos,voco Gold Coast,35 609 038 502,22180125712401UDWM09,None,...,parsed,2026-04-24T04:30:27Z,2026-04-26T13:16:42.001054Z,100793,"Burger patty, wagyu beef, Laneway, low p",4.000,piece/unit,71.26,None,285.03
1,2,C:\git\ms_outlook\attachments\2026-04-24_topcu...,2,AQMkADAwATNiZmYAZC1kNTczLTMyOTgtMDACLTAwCgBGAA...,Paulo.Penecitos@topcut.com.au,Paulo Penecitos,voco Gold Coast,35 609 038 502,22180125712401UDWM09,None,...,parsed,2026-04-24T04:30:27Z,2026-04-26T13:16:42.001054Z,500873,"Lamb, diced, 25x25mm, multiple wrap, va",15.000,kilogram,28.00,None,420.00
2,19,C:\git\ms_outlook\attachments\2026-04-24_topcu...,1,AQMkADAwATNiZmYAZC1kNTczLTMyOTgtMDACLTAwCgBGAA...,Paulo.Penecitos@topcut.com.au,Paulo Penecitos,voco Gold Coast,35 609 038 502,22180125712401UDWM09,None,...,parsed,2026-04-24T04:30:27Z,2026-04-26T12:56:41.106828Z,100793,"Burger patty, wagyu beef, Laneway, low p",4.000,piece/unit,71.26,None,285.03
3,20,C:\git\ms_outlook\attachments\2026-04-24_topcu...,2,AQMkADAwATNiZmYAZC1kNTczLTMyOTgtMDACLTAwCgBGAA...,Paulo.Penecitos@topcut.com.au,Paulo Penecitos,voco Gold Coast,35 609 038 502,22180125712401UDWM09,None,...,parsed,2026-04-24T04:30:27Z,2026-04-26T12:56:41.106828Z,500873,"Lamb, diced, 25x25mm, multiple wrap, va",15.000,kilogram,28.00,None,420.00
4,40,C:\git\ms_outlook\attachments\2026-04-24_topcu...,1,AQMkADAwATNiZmYAZC1kNTczLTMyOTgtMDACLTAwCgBGAA...,Paulo.Penecitos@topcut.com.au,Paulo Penecitos,voco Gold Coast,35 609 038 502,22180125712401UDWM09,None,...,parsed,2026-04-24T04:30:27Z,2026-04-26T12:40:14.318124Z,100793,"Burger patty, wagyu beef, Laneway, low p",4.000,piece/unit,71.26,None,285.03
5,41,C:\git\ms_outlook\attachments\2026-04-24_topcu...,2,AQMkADAwATNiZmYAZC1kNTczLTMyOTgtMDACLTAwCgBGAA...,Paulo.Penecitos@topcut.com.au,Paulo Penecitos,voco Gold Coast,35 609 038 502,22180125712401UDWM09,None,...,parsed,2026-04-24T04:30:27Z,2026-04-26T12:40:14.318124Z,500873,"Lamb, diced, 25x25mm, multiple wrap, va",15.000,kilogram,28.00,None,420.00
